# ScholarRAG: chunk-size experiment

**Question:** does a smaller chunk improve retrieval? At 350 words, 22.5% of chunks are longer than the 512-token limit of `bge-small-en-v1.5`, so the dense retriever never sees their tail (section 6.9 of `ScholarRAG_local.ipynb`).

| Setting | Value |
|---|---|
| Chunk sizes | 350 (baseline) · 250 · 180 words |
| Overlap | 60 words for every size: still longer than the longest evidence quote (37 words), so every quote fits inside at least one chunk |
| Everything else | Same as the baseline: parser, paragraph-boundary cut, bibliography filter, `bge-small-en-v1.5`, BM25 |
| Questions | Set A (64 LLM questions, `questions_v1.csv`) and set B1 (15 handwritten answerable questions), reported per source |

Gold chunks are found from the evidence quotes, so the same frozen questions work for every chunk size.

**How to read Hit@k across sizes:** smaller chunks mean more chunks in the index, so a chunk-level hit is harder by construction. `PaperHit@10` (is the right *paper* in the top 10) does not depend on chunk count and is the fairer cross-size comparison.

## 0. Setup

In [1]:
import re, json, unicodedata
import numpy as np, pandas as pd
from pathlib import Path
import pymupdf

BASE = Path.home() / "Projects" / "scholar-rag" / "data"
SIZES = [350, 250, 180]
OVERLAP = 60

meta = {}
for line in open(BASE / "selected.jsonl"):
    r = json.loads(line)
    meta[r["id"].replace("/", "_")] = r["title"]

# set A + bagian terjawab dari set B
qa = pd.read_csv(BASE / "eval" / "questions_v1.csv", dtype={"paper_id": str})
qa = qa[["id", "question", "evidence", "paper_id"]].assign(source="llm")
hw = pd.read_csv(BASE / "eval" / "questions_handwritten_v1.csv", dtype=str).fillna("")
hb = hw[hw.answerable.str.upper() == "TRUE"][["id", "question", "evidence", "paper_id"]].assign(source="handwritten")
qs = pd.concat([qa, hb], ignore_index=True)
print(qs.source.value_counts().to_dict())

{'llm': 64, 'handwritten': 15}


## 1. Build chunks at three sizes

`extract_units`, `chunk_units` and `looks_like_refs` are copied unchanged from sections 3 and 6.2 of the main notebook. PDFs are parsed once and re-chunked per size.

In [2]:
REF_RE = re.compile(r"^\s*(\d+\.?\s*)?(references|bibliography)\s*$", re.I)

def extract_units(pdf_path):
    """Kembalikan list (page, paragraf) dengan References sudah dibuang."""
    doc = pymupdf.open(pdf_path)
    units = []
    ref_at = None
    for pno, page in enumerate(doc, start=1):
        for b in page.get_text("blocks"):
            if b[6] != 0:
                continue
            t = b[4]
            t = re.sub(r"-\n(\w)", r"\1", t)
            t = re.sub(r"\s+", " ", t).strip()
            if len(t) < 40:
                if REF_RE.match(t) and pno > len(doc) * 0.4:
                    ref_at = len(units)
                continue
            letters = sum(ch.isalpha() for ch in t)
            if letters / len(t) < 0.5:
                continue
            units.append((pno, t))
    return units[:ref_at] if ref_at is not None else units

def chunk_units(units, target=350, overlap=60):
    words, pages, boundaries = [], [], set()
    for pno, t in units:
        boundaries.add(len(words))
        w = t.split()
        words += w
        pages += [pno] * len(w)
    n = len(words)
    chunks, start = [], 0
    while start < n:
        end = min(start + target, n)
        if end < n:
            cands = [b for b in boundaries if start + int(0.75 * target) < b <= start + target]
            if cands:
                end = max(cands)
        chunks.append((" ".join(words[start:end]), pages[start], pages[end - 1]))
        if end >= n:
            break
        start = max(end - overlap, start + 1)
    return chunks

def looks_like_refs(t):
    years = len(re.findall(r"\b(?:19|20)\d{2}[a-z]?\b", t))
    initials = len(re.findall(r"\b[A-Z]\.", t))
    return (years >= 5 and initials >= 8) or t.count("arXiv preprint") >= 2

units = {pdf.stem: extract_units(pdf) for pdf in sorted((BASE / "raw").glob("*.pdf"))}
print(len(units), "PDF diparse")

200 PDF diparse


In [3]:
chunk_sets, stats = {}, []
for size in SIZES:
    rows = []
    for pid, u in units.items():
        for i, (text, p0, p1) in enumerate(chunk_units(u, target=size, overlap=OVERLAP)):
            rows.append({"chunk_id": f"{pid}_c{i:04d}", "paper_id": pid, "title": meta.get(pid, ""),
                         "page": p0, "page_end": p1, "text": text, "n_words": len(text.split())})
    df = pd.DataFrame(rows)
    is_ref = df.text.apply(looks_like_refs)
    clean = df[~is_ref].reset_index(drop=True)
    clean.to_json(BASE / f"chunks_{size}.jsonl", orient="records", lines=True, force_ascii=False)
    chunk_sets[size] = clean
    stats.append({"size": size, "chunks": len(df), "bibliography removed": int(is_ref.sum()),
                  "indexed": len(clean), "median words": int(clean.n_words.median())})

# ukuran 350 harus sama persis dengan chunks_clean.jsonl dari notebook utama
base_ids = pd.read_json(BASE / "chunks_clean.jsonl", lines=True).chunk_id.tolist()
print("350 sama dengan chunks_clean.jsonl:", chunk_sets[350].chunk_id.tolist() == base_ids)
pd.DataFrame(stats).set_index("size")

350 sama dengan chunks_clean.jsonl: True


,chunks,bibliography removed,indexed,median words
size,,,,
350,3991,81,3910,331
250,6054,109,5945,240
180,9532,153,9379,176


## 2. Gold chunks from evidence quotes

In [4]:
def norm(s):
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"-\s+(\w)", r"\1", s)
    return re.sub(r"\s+", " ", s).strip().lower()

def gold_for(row, ch):
    g = set()
    pids = [p.strip() for p in row.paper_id.split(";") if p.strip()]
    evs = [e.strip() for e in row.evidence.split(";") if e.strip()] if row.source == "handwritten" else [row.evidence]
    for pid, ev in zip(pids, evs):
        m = (ch.paper_id == pid) & ch.norm.str.contains(norm(ev), regex=False)
        g |= set(ch.index[m])
    return g

golds = {}
for size, ch in chunk_sets.items():
    ch["norm"] = ch.text.map(norm)
    golds[size] = {r.id: gold_for(r, ch) for _, r in qs.iterrows()}
    missing = [i for i, g in golds[size].items() if not g]
    print(f"{size}: soal tanpa gold = {missing or 'tidak ada'}")

qs["paper_id"] = qs.paper_id.str.split(";").str[0]   # PaperHit memakai paper pertama

350: soal tanpa gold = tidak ada


250: soal tanpa gold = tidak ada


180: soal tanpa gold = tidak ada


## 3. Dense and BM25 indexes

Dense embeddings are cached per size in `index/emb_bge_small_<size>.npy`; size 350 reuses the main notebook's cache when the chunk ids match. The truncation column is the share of chunks longer than the model's 512-token limit.

In [5]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
QP = "Represent this sentence for searching relevant passages: "

def tok(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in ENGLISH_STOP_WORDS]

def load_or_embed(size, ch):
    ids = ch.chunk_id.tolist()
    cands = [(BASE / "index" / f"emb_bge_small_{size}.npy", BASE / "index" / f"chunk_ids_{size}.json")]
    if size == 350:
        cands.append((BASE / "index" / "emb_bge_small.npy", BASE / "index" / "chunk_ids.json"))
    for emb_p, ids_p in cands:
        if emb_p.exists() and ids_p.exists() and json.load(open(ids_p)) == ids:
            return np.load(emb_p)
    E = model.encode(ch.text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    np.save(cands[0][0], E); json.dump(ids, open(cands[0][1], "w"))
    return E

indexes, trunc = {}, {}
for size, ch in chunk_sets.items():
    E = load_or_embed(size, ch)
    bm25 = BM25Okapi([tok(t) for t in ch.text])
    indexes[size] = (E, bm25)
    lens = np.array([len(model.tokenizer(t, truncation=False)["input_ids"]) for t in ch.text])
    trunc[size] = (lens > model.max_seq_length).mean()
    print(f"{size}: {E.shape[0]} chunk | median {int(np.median(lens))} token | terpotong {trunc[size]:.1%}")

W1019 15:49:24.362000 18538 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (527 > 512). Running this sequence through the model will result in indexing errors


350: 3910 chunk | median 471 token | terpotong 22.5%


250: 5945 chunk | median 339 token | terpotong 0.5%


180: 9379 chunk | median 247 token | terpotong 0.0%


## 4. Results

In [6]:
def evaluate(size, retriever, ks=(1, 5, 10, 20)):
    ch, (E, bm25), gold = chunk_sets[size], indexes[size], golds[size]
    Q = model.encode([QP + q for q in qs.question], normalize_embeddings=True) if retriever == "Dense" else None
    rows = []
    for j, q in enumerate(qs.itertuples()):
        scores = E @ Q[j] if retriever == "Dense" else bm25.get_scores(tok(q.question))
        ranked = np.argsort(-scores)[:max(ks)]
        r_chunk = next((i + 1 for i, c in enumerate(ranked) if c in gold[q.id]), np.nan)
        papers = ch.paper_id.values[ranked]
        r_paper = next((i + 1 for i, p in enumerate(papers) if p == q.paper_id), np.nan)
        rows.append((q.id, q.source, r_chunk, r_paper))
    return pd.DataFrame(rows, columns=["id", "source", "rank_chunk", "rank_paper"])

def summarize(res):
    out = {}
    for src, d in res.groupby("source"):
        out[src] = {"n": len(d), "Hit@1": (d.rank_chunk <= 1).mean(), "Hit@5": (d.rank_chunk <= 5).mean(),
                    "Hit@10": (d.rank_chunk <= 10).mean(), "MRR": (1 / d.rank_chunk).fillna(0).mean(),
                    "PaperHit@10": (d.rank_paper <= 10).mean()}
    return out

all_res, table = {}, []
for size in SIZES:
    for retriever in ["Dense", "BM25"]:
        res = evaluate(size, retriever)
        all_res[(size, retriever)] = res
        for src, m in summarize(res).items():
            table.append({"source": src, "size": size, "retriever": retriever, **m})

results = pd.DataFrame(table).set_index(["source", "retriever", "size"]).sort_index(ascending=[False, True, False])
results.to_csv(BASE / "eval" / "results_chunk_size.csv")
results.round(3)

n  Hit@1  Hit@5  Hit@10    MRR  PaperHit@10
source      retriever size                                              
llm         BM25      350   64  0.625  0.906   0.938  0.751        1.000
                      250   64  0.578  0.891   0.906  0.708        1.000
                      180   64  0.547  0.828   0.922  0.659        1.000
            Dense     350   64  0.391  0.656   0.797  0.512        0.922
                      250   64  0.359  0.641   0.688  0.475        0.891
                      180   64  0.344  0.734   0.781  0.501        0.953
handwritten BM25      350   15  0.200  0.800   0.933  0.442        1.000
                      250   15  0.333  0.867   0.867  0.508        1.000
                      180   15  0.400  0.800   0.867  0.588        1.000
            Dense     350   15  0.600  0.733   0.800  0.665        1.000
                      250   15  0.533  0.733   0.800  0.624        1.000
                      180   15  0.533  0.733   0.800  0.639        1.000

### 4.1 Dense truncation vs. chunk size

In [7]:
pd.DataFrame({"chunks": {s: len(chunk_sets[s]) for s in SIZES},
              "truncated (>512 tokens)": {s: f"{trunc[s]:.1%}" for s in SIZES}})

,chunks,truncated (>512 tokens)
350,3910,22.5%
250,5945,0.5%
180,9379,0.0%


## 5. Reading the results

**The 350-word baseline reproduces exactly** (same chunk ids as `chunks_clean.jsonl`, same scores as sections 6 and 7.5 of the main notebook), so the three sizes are compared on equal terms.

**1. Removing truncation did not help the dense retriever.** Truncation falls from 22.5% (350) to 0.5% (250) and 0% (180), but dense MRR on set A goes 0.512 → 0.475 → 0.501 and Hit@1 0.391 → 0.359 → 0.344. Only PaperHit@10 at 180 words is higher (0.953 vs 0.922, two questions). The cut-off tail of long chunks was not the main reason dense trails BM25.

**2. BM25 moves in opposite directions on the two question sets.**
- Set A (LLM, n = 64): BM25 gets worse as chunks shrink: MRR 0.751 → 0.708 → 0.659, Hit@1 0.625 → 0.547 (5 questions).
- Set B1 (handwritten, n = 15): BM25 gets better: MRR 0.442 → 0.508 → 0.588, Hit@1 0.200 → 0.400 (3 more questions).

A plausible reading: LLM questions copy many words from their passage, so a long chunk that contains all of them scores high; paraphrased questions share only a few words, and those few words count for more in a short chunk. With n = 15, one question is 6.7 points, so the set B trend is a hint, not a result.

**3. Dense stays ahead on handwritten questions at every size** (Hit@1 0.53–0.60 vs BM25 0.20–0.40), confirming the per-source result of section 7.5.

**4. Smaller chunks slightly ease the metric:** more questions have two gold chunks (13 at 350, 22 at 180 words), because the 60-word overlap is a larger share of a short chunk. Scores still did not improve on set A.

### Decision

- **Keep 350 words as the default index.** No size wins on both sets; 350 is best for both retrievers on set A and for dense on set B1.
- **Revisit with the reranker in M3.** The cross-encoder reads question + passage within 512 tokens. At 350 words the median chunk is already 471 tokens, so many pairs would be cut; at 250 words (median 339 tokens, 0.5% over the limit) they fit. Hybrid + rerank should be tested on both 350 and 250.